# Perf Logs: Query-Level Comparative Analysis

This notebook analyzes perf-stat logs under `./logs/perf_logs` using the new grouped file layout.

It will:
- Recursively load all `*.perfstat.csv` files from `./logs/perf_logs`
- Parse run and query metadata from file names and metadata sidecar files
- Keep analysis at **individual query** granularity (no chunk-level aggregation assumptions)
- Compare queries for each event
- Compare the same query/event across PCs (when multiple PCs are present)

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

base_dir = Path("../runner/logs/perf_logs")
perf_files = sorted(base_dir.rglob("*.perfstat.csv"))
meta_files = sorted(base_dir.rglob("*__perf_metadata.txt"))

print(f"Found {len(perf_files):,} perfstat files under {base_dir}")
print(f"Found {len(meta_files):,} metadata files under {base_dir}")

# Supported file formats (new + backward-compatible):
# 1) RUNID__QUERY.sql.perfstat.csv
# 2) RUNID__QUERY.sql__CHUNK_REPEAT.perfstat.csv (legacy)
# 3) PCID__RUNID__QUERY.sql.perfstat.csv (optional grouped-by-PC naming)
patterns = [
    re.compile(r"^(?P<run_id>[A-F0-9]+)__(?P<query_file>.+?)\.perfstat\.csv$") ,
    re.compile(r"^(?P<run_id>[A-F0-9]+)__(?P<query_file>.+?)__(?P<chunk>\d+)_(?P<repeat>\d+)\.perfstat\.csv$"),
    re.compile(r"^(?P<pc_id>[A-Za-z0-9_-]+)__(?P<run_id>[A-F0-9]+)__(?P<query_file>.+?)\.perfstat\.csv$"),
]

col_names = [
    "value_raw",
    "unit",
    "event",
    "runtime_count",
    "runtime_pct",
    "ratio",
    "ratio_desc",
]

def parse_key_value_file(path: Path) -> dict:
    out = {}
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip()
    return out

run_meta = {}
for meta_path in meta_files:
    m = re.match(r"^(?P<run_id>[A-F0-9]+)__perf_metadata\.txt$", meta_path.name)
    if not m:
        continue
    rid = m.group("run_id")
    run_meta[rid] = parse_key_value_file(meta_path)

frames = []
parse_failures = []

for fp in perf_files:
    parsed = None
    for pat in patterns:
        m = pat.match(fp.name)
        if m:
            parsed = m.groupdict()
            break

    if parsed is None:
        parse_failures.append(fp.name)
        continue

    rid = parsed.get("run_id", "")
    meta = run_meta.get(rid, {})
    df = pd.read_csv(
        fp,
        comment="#",
        header=None,
        names=col_names,
        on_bad_lines="skip",
    )

    if df.empty:
        continue

    n_events = max(df["event"].nunique(dropna=True), 1)
    df["sample_idx"] = (np.arange(len(df)) // n_events) + 1

    df["run_id"] = rid
    df["pc_id"] = rid
    df["query_file"] = parsed.get("query_file", "")
    df["query_name"] = df["query_file"].str.replace(r"\.sql$", "", regex=True)
    df["chunk"] = pd.to_numeric(parsed.get("chunk"), errors="coerce")
    df["repeat"] = pd.to_numeric(parsed.get("repeat"), errors="coerce")
    df["run_started_utc"] = meta.get("run_started_utc")
    df["query_filter"] = meta.get("query_filter")
    df["source_file"] = str(fp.relative_to(base_dir))

    frames.append(df)

perf_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=col_names)

for c in ["value_raw", "runtime_count", "runtime_pct", "ratio", "chunk", "repeat", "sample_idx"]:
    if c in perf_df.columns:
        perf_df[c] = pd.to_numeric(perf_df[c], errors="coerce")

print(f"Loaded rows: {len(perf_df):,}")
print(f"Filename parse failures: {len(parse_failures)}")
if parse_failures:
    print("Examples:", parse_failures[:5])

perf_df.head(10)

In [ ]:
# Dataset quality + individual-query coverage
summary = {
    "rows": len(perf_df),
    "columns": perf_df.shape[1],
    "n_pcs": perf_df["pc_id"].nunique(dropna=True),
    "n_run_ids": perf_df["run_id"].nunique(dropna=True),
    "n_queries": perf_df["query_name"].nunique(dropna=True),
    "n_events": perf_df["event"].nunique(dropna=True),
    "n_source_files": perf_df["source_file"].nunique(dropna=True),
}

print("Dataset summary:")
display(pd.Series(summary, name="value").to_frame())

print("\nCoverage by PC and query (number of sample runs):")
coverage = (
    perf_df.groupby(["pc_id", "query_name"], dropna=False)["sample_idx"]
    .nunique()
    .rename("n_samples")
    .reset_index()
    .sort_values(["pc_id", "query_name"])
)
display(coverage)

print("\nTop events by row count:")
display(perf_df["event"].value_counts().rename_axis("event").to_frame("rows"))

print("\nMissingness (%):")
display((perf_df.isna().mean() * 100).round(2).sort_values(ascending=False).rename("pct_missing").to_frame())

In [ ]:
# Query-level comparison (always per individual query)
core_events = [
    "cycles",
    "instructions",
    "branches",
    "branch-misses",
    "cache-references",
    "cache-misses",
    "context-switches",
    "cpu-migrations",
    "page-faults",
]

query_event_stats = (
    perf_df.groupby(["pc_id", "query_name", "event"], dropna=False)["value_raw"]
    .agg(count="count", mean_value="mean", median_value="median", std_value="std", p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)

print("Per-query stats (long format):")
display(
    query_event_stats.sort_values(["pc_id", "query_name", "event"]).reset_index(drop=True)
 )

print("\nQuery comparison table (median value by event within each PC):")
query_compare = (
    query_event_stats[query_event_stats["event"].isin(core_events)]
    .pivot_table(index="query_name", columns=["pc_id", "event"], values="median_value")
    .sort_index()
)
display(query_compare)

print("\nQuery ranking by median cycles (lower is better):")
cycles_rank = (
    query_event_stats[query_event_stats["event"] == "cycles"]
    .sort_values(["pc_id", "median_value", "query_name"])[["pc_id", "query_name", "median_value", "std_value", "count"]]
    .rename(columns={"median_value": "cycles_median", "std_value": "cycles_std"})
)
display(cycles_rank)

In [ ]:
# Cross-PC comparison for the same query/event
pc_count = perf_df["pc_id"].nunique(dropna=True)

if pc_count < 2:
    print("Only one PC found in current dataset; cross-PC comparison needs >= 2 PCs.")
else:
    cross_pc = (
        perf_df.groupby(["query_name", "event", "pc_id"], dropna=False)["value_raw"]
        .median()
        .rename("median_value")
        .reset_index()
    )

    cross_pc_pivot = cross_pc.pivot_table(
        index=["query_name", "event"],
        columns="pc_id",
        values="median_value"
    )

    pc_cols = list(cross_pc_pivot.columns)
    cross_pc_pivot["pc_max"] = cross_pc_pivot[pc_cols].max(axis=1)
    cross_pc_pivot["pc_min"] = cross_pc_pivot[pc_cols].min(axis=1)
    cross_pc_pivot["pc_max_over_min"] = np.where(
        cross_pc_pivot["pc_min"] > 0,
        cross_pc_pivot["pc_max"] / cross_pc_pivot["pc_min"],
        np.nan,
    )

    print("Largest cross-PC spread by query/event (median max/min):")
    display(cross_pc_pivot.sort_values("pc_max_over_min", ascending=False).head(40))

    print("\nCross-PC spread for core events only:")
    core_spread = cross_pc_pivot.reset_index()
    core_spread = core_spread[core_spread["event"].isin([
        "cycles", "instructions", "cache-misses", "branch-misses", "page-faults"
    ])]
    display(core_spread.sort_values(["event", "pc_max_over_min"], ascending=[True, False]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set_style("whitegrid")
sns.set_palette("Set2")

# Prepare data for plotting
plot_events = ["cycles", "instructions", "cache-misses", "branch-misses", "page-faults"]
plot_df = perf_df[perf_df["event"].isin(plot_events)].dropna(subset=["value_raw", "query_name", "event"])
query_order = sorted(plot_df["query_name"].dropna().unique())

def _query_family(name: str) -> str:
    return re.sub(r"-query[AB]$", "", str(name))

def add_ab_group_separators(ax, labels):
    labels = list(labels)
    for i in range(len(labels) - 1):
        if _query_family(labels[i]) != _query_family(labels[i + 1]):
            ax.axvline(i + 0.5, color="gray", linestyle=":", linewidth=1.2, alpha=0.8)

# Boxplot: Cycles
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "cycles"]

if not event_data.empty:
    sns.boxplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        hue="pc_id",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Cycles by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)
    ax.legend(title="PC", fontsize=9, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplot (cycles) created")

In [ ]:
# Boxplot: Instructions
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "instructions"]

if not event_data.empty:
    sns.boxplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        hue="pc_id",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Instructions by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)
    ax.legend(title="PC", fontsize=9, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplot (instructions) created")

In [ ]:
# Boxplot: Cache Misses
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "cache-misses"]

if not event_data.empty:
    sns.boxplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        hue="pc_id",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Cache Misses by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)
    ax.legend(title="PC", fontsize=9, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplot (cache-misses) created")

In [ ]:
# Boxplot: Branch Misses
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "branch-misses"]

if not event_data.empty:
    sns.boxplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        hue="pc_id",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Branch Misses by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)
    ax.legend(title="PC", fontsize=9, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplot (branch-misses) created")

In [ ]:
# Boxplot: Page Faults
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "page-faults"]

if not event_data.empty:
    sns.boxplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        hue="pc_id",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Page Faults by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)
    ax.legend(title="PC", fontsize=9, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplot (page-faults) created")

In [ ]:
# Boxplots: Memory/Cache Events
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Boxplots: Memory/Cache Events by Query", fontsize=14, fontweight="bold")

memory_events = ["cache-misses", "branch-misses", "page-faults"]
for idx, event in enumerate(memory_events):
    ax = axes[idx]
    event_data = plot_df[plot_df["event"] == event]

    if not event_data.empty:
        sns.boxplot(
            data=event_data,
            x="query_name",
            y="value_raw",
            hue="pc_id",
            order=query_order,
            ax=ax,
            palette="Set2"
        )
        ax.set_title(event, fontsize=12, fontweight="bold")
        ax.set_xlabel("Query", fontsize=10)
        ax.set_ylabel("Count", fontsize=10)
        ax.tick_params(axis="x", rotation=45)
        add_ab_group_separators(ax, query_order)
        ax.legend(title="PC", fontsize=8, loc="best")

plt.tight_layout()
plt.show()

print("✓ Boxplots (memory/cache events) created")

In [ ]:
# Violin plot: Cycles
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "cycles"]

if not event_data.empty:
    sns.violinplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Cycles Distribution by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plot (cycles) created")

In [ ]:
# Violin plot: Instructions
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "instructions"]

if not event_data.empty:
    sns.violinplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Instructions Distribution by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plot (instructions) created")

In [ ]:
# Violin plot: Cache Misses
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "cache-misses"]

if not event_data.empty:
    sns.violinplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Cache Misses Distribution by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plot (cache-misses) created")

In [ ]:
# Violin plot: Branch Misses
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "branch-misses"]

if not event_data.empty:
    sns.violinplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Branch Misses Distribution by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plot (branch-misses) created")

In [ ]:
# Violin plot: Page Faults
fig, ax = plt.subplots(figsize=(14, 6))
event_data = plot_df[plot_df["event"] == "page-faults"]

if not event_data.empty:
    sns.violinplot(
        data=event_data,
        x="query_name",
        y="value_raw",
        order=query_order,
        ax=ax,
        palette="Set2"
    )
    ax.set_title("Page Faults Distribution by Query", fontsize=13, fontweight="bold")
    ax.set_xlabel("Query", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.tick_params(axis="x", rotation=45)
    add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plot (page-faults) created")

In [ ]:
# Violin plots: Memory/Cache Events
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Violin Plots: Memory/Cache Events Distribution", fontsize=14, fontweight="bold")

memory_events = ["cache-misses", "branch-misses", "page-faults"]
for idx, event in enumerate(memory_events):
    ax = axes[idx]
    event_data = plot_df[plot_df["event"] == event]

    if not event_data.empty:
        sns.violinplot(
            data=event_data,
            x="query_name",
            y="value_raw",
            order=query_order,
            ax=ax,
            palette="Set2"
        )
        ax.set_title(event, fontsize=11, fontweight="bold")
        ax.set_xlabel("Query", fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        add_ab_group_separators(ax, query_order)

plt.tight_layout()
plt.show()

print("✓ Violin plots (memory/cache events) created")

In [ ]:
# Cross-PC spread heatmap: max/min ratio for each query/event
if pc_count >= 2:
    spread_data = core_spread[["query_name", "event", "pc_max_over_min"]].copy()
    spread_pivot = spread_data.pivot_table(
        index="query_name",
        columns="event",
        values="pc_max_over_min"
    )
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        spread_pivot,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn_r",
        cbar_kws={"label": "Max/Min Ratio"},
        ax=ax
    )
    ax.set_title("Cross-PC Spread by Query/Event (Median values)\nRed = Higher spread (less consistent across PCs)", 
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Event", fontsize=10)
    ax.set_ylabel("Query", fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Cross-PC spread heatmap created")
else:
    print("Skipped cross-PC spread heatmap (need >= 2 PCs)")

In [ ]:
# Query ranking: Cycles
fig, ax = plt.subplots(figsize=(14, 6))

event_stats = query_event_stats[query_event_stats["event"] == "cycles"].copy()
event_stats = event_stats.sort_values("median_value", ascending=True)

if not event_stats.empty:
    x_pos = np.arange(len(event_stats))
    
    for pc_id in event_stats["pc_id"].unique():
        pc_data = event_stats[event_stats["pc_id"] == pc_id]
        ax.errorbar(
            x_pos[:len(pc_data)],
            pc_data["mean_value"],
            yerr=pc_data["std_value"],
            fmt="o-",
            label=f"PC: {pc_id}",
            alpha=0.7,
            capsize=5,
            linewidth=2
        )
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=9)
    ax.set_title("Cycles: Query Rankings (Mean ± Std)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Count (±std)", fontsize=11)
    ax.set_xlabel("Query", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query ranking (cycles) created")

In [ ]:
# Query ranking: Instructions
fig, ax = plt.subplots(figsize=(14, 6))

event_stats = query_event_stats[query_event_stats["event"] == "instructions"].copy()
event_stats = event_stats.sort_values("median_value", ascending=True)

if not event_stats.empty:
    x_pos = np.arange(len(event_stats))
    
    for pc_id in event_stats["pc_id"].unique():
        pc_data = event_stats[event_stats["pc_id"] == pc_id]
        ax.errorbar(
            x_pos[:len(pc_data)],
            pc_data["mean_value"],
            yerr=pc_data["std_value"],
            fmt="o-",
            label=f"PC: {pc_id}",
            alpha=0.7,
            capsize=5,
            linewidth=2
        )
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=9)
    ax.set_title("Instructions: Query Rankings (Mean ± Std)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Count (±std)", fontsize=11)
    ax.set_xlabel("Query", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query ranking (instructions) created")

In [ ]:
# Query ranking: Cache Misses
fig, ax = plt.subplots(figsize=(14, 6))

event_stats = query_event_stats[query_event_stats["event"] == "cache-misses"].copy()
event_stats = event_stats.sort_values("median_value", ascending=True)

if not event_stats.empty:
    x_pos = np.arange(len(event_stats))
    
    for pc_id in event_stats["pc_id"].unique():
        pc_data = event_stats[event_stats["pc_id"] == pc_id]
        ax.errorbar(
            x_pos[:len(pc_data)],
            pc_data["mean_value"],
            yerr=pc_data["std_value"],
            fmt="o-",
            label=f"PC: {pc_id}",
            alpha=0.7,
            capsize=5,
            linewidth=2
        )
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=9)
    ax.set_title("Cache Misses: Query Rankings (Mean ± Std)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Count (±std)", fontsize=11)
    ax.set_xlabel("Query", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query ranking (cache-misses) created")

In [ ]:
# Query ranking: Branch Misses
fig, ax = plt.subplots(figsize=(14, 6))

event_stats = query_event_stats[query_event_stats["event"] == "branch-misses"].copy()
event_stats = event_stats.sort_values("median_value", ascending=True)

if not event_stats.empty:
    x_pos = np.arange(len(event_stats))
    
    for pc_id in event_stats["pc_id"].unique():
        pc_data = event_stats[event_stats["pc_id"] == pc_id]
        ax.errorbar(
            x_pos[:len(pc_data)],
            pc_data["mean_value"],
            yerr=pc_data["std_value"],
            fmt="o-",
            label=f"PC: {pc_id}",
            alpha=0.7,
            capsize=5,
            linewidth=2
        )
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=9)
    ax.set_title("Branch Misses: Query Rankings (Mean ± Std)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Count (±std)", fontsize=11)
    ax.set_xlabel("Query", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query ranking (branch-misses) created")

In [ ]:
# Query ranking: Page Faults
fig, ax = plt.subplots(figsize=(14, 6))

event_stats = query_event_stats[query_event_stats["event"] == "page-faults"].copy()
event_stats = event_stats.sort_values("median_value", ascending=True)

if not event_stats.empty:
    x_pos = np.arange(len(event_stats))
    
    for pc_id in event_stats["pc_id"].unique():
        pc_data = event_stats[event_stats["pc_id"] == pc_id]
        ax.errorbar(
            x_pos[:len(pc_data)],
            pc_data["mean_value"],
            yerr=pc_data["std_value"],
            fmt="o-",
            label=f"PC: {pc_id}",
            alpha=0.7,
            capsize=5,
            linewidth=2
        )
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=9)
    ax.set_title("Page Faults: Query Rankings (Mean ± Std)", fontsize=13, fontweight="bold")
    ax.set_ylabel("Mean Count (±std)", fontsize=11)
    ax.set_xlabel("Query", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query ranking (page-faults) created")

In [ ]:
# Query ranking with error bars: Memory/Cache Events
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Query Rankings (Mean ± Std): Memory/Cache Events", fontsize=14, fontweight="bold")

memory_ranking = ["cache-misses", "branch-misses", "page-faults"]

for idx, event in enumerate(memory_ranking):
    ax = axes[idx]
    
    event_stats = query_event_stats[query_event_stats["event"] == event].copy()
    event_stats = event_stats.sort_values("median_value", ascending=True)
    
    if not event_stats.empty:
        x_pos = np.arange(len(event_stats))
        
        for pc_id in event_stats["pc_id"].unique():
            pc_data = event_stats[event_stats["pc_id"] == pc_id]
            ax.errorbar(
                x_pos[:len(pc_data)],
                pc_data["mean_value"],
                yerr=pc_data["std_value"],
                fmt="o-",
                label=f"PC: {pc_id}",
                alpha=0.7,
                capsize=5
            )
        
        ax.set_xticks(x_pos)
        ax.set_xticklabels(event_stats["query_name"], rotation=45, ha="right", fontsize=8)
        ax.set_title(event, fontsize=11, fontweight="bold")
        ax.set_ylabel("Mean Count (±std)", fontsize=9)
        ax.set_xlabel("Query", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Query rankings (memory/cache events) created")

In [ ]:
import matplotlib.gridspec as gridspec

# Faceted grid: Cycles across all queries and PCs
fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, len(plot_df["query_name"].unique()), hspace=0.3, wspace=0.3)

queries_sorted = sorted(plot_df["query_name"].unique())

for q_idx, query in enumerate(queries_sorted):
    ax = fig.add_subplot(gs[0, q_idx])
    
    subset = plot_df[(plot_df["event"] == "cycles") & (plot_df["query_name"] == query)]
    
    if not subset.empty:
        sns.boxplot(
            data=subset,
            x="pc_id",
            y="value_raw",
            ax=ax,
            palette="Set2"
        )
        ax.set_title(f"{query}", fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Cycles" if q_idx == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Cycles: All Queries × PCs (Boxplot)", fontsize=13, fontweight="bold", y=1.00)
plt.show()

print("✓ Faceted grid (cycles) created")

In [ ]:
# Faceted grid: Instructions across all queries and PCs
fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, len(queries_sorted), hspace=0.3, wspace=0.3)

for q_idx, query in enumerate(queries_sorted):
    ax = fig.add_subplot(gs[0, q_idx])
    
    subset = plot_df[(plot_df["event"] == "instructions") & (plot_df["query_name"] == query)]
    
    if not subset.empty:
        sns.boxplot(
            data=subset,
            x="pc_id",
            y="value_raw",
            ax=ax,
            palette="Set2"
        )
        ax.set_title(f"{query}", fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Instructions" if q_idx == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Instructions: All Queries × PCs (Boxplot)", fontsize=13, fontweight="bold", y=1.00)
plt.show()

print("✓ Faceted grid (instructions) created")

In [ ]:
# Faceted grid: Cache Misses across all queries and PCs
fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, len(queries_sorted), hspace=0.3, wspace=0.3)

for q_idx, query in enumerate(queries_sorted):
    ax = fig.add_subplot(gs[0, q_idx])
    
    subset = plot_df[(plot_df["event"] == "cache-misses") & (plot_df["query_name"] == query)]
    
    if not subset.empty:
        sns.boxplot(
            data=subset,
            x="pc_id",
            y="value_raw",
            ax=ax,
            palette="Set2"
        )
        ax.set_title(f"{query}", fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Cache Misses" if q_idx == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Cache Misses: All Queries × PCs (Boxplot)", fontsize=13, fontweight="bold", y=1.00)
plt.show()

print("✓ Faceted grid (cache-misses) created")

In [ ]:
# Faceted grid: Branch Misses across all queries and PCs
fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, len(queries_sorted), hspace=0.3, wspace=0.3)

for q_idx, query in enumerate(queries_sorted):
    ax = fig.add_subplot(gs[0, q_idx])
    
    subset = plot_df[(plot_df["event"] == "branch-misses") & (plot_df["query_name"] == query)]
    
    if not subset.empty:
        sns.boxplot(
            data=subset,
            x="pc_id",
            y="value_raw",
            ax=ax,
            palette="Set2"
        )
        ax.set_title(f"{query}", fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Branch Misses" if q_idx == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Branch Misses: All Queries × PCs (Boxplot)", fontsize=13, fontweight="bold", y=1.00)
plt.show()

print("✓ Faceted grid (branch-misses) created")

In [ ]:
# Faceted grid: Page Faults across all queries and PCs
fig = plt.figure(figsize=(16, 5))
gs = gridspec.GridSpec(1, len(queries_sorted), hspace=0.3, wspace=0.3)

for q_idx, query in enumerate(queries_sorted):
    ax = fig.add_subplot(gs[0, q_idx])
    
    subset = plot_df[(plot_df["event"] == "page-faults") & (plot_df["query_name"] == query)]
    
    if not subset.empty:
        sns.boxplot(
            data=subset,
            x="pc_id",
            y="value_raw",
            ax=ax,
            palette="Set2"
        )
        ax.set_title(f"{query}", fontsize=10, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Page Faults" if q_idx == 0 else "", fontsize=9)
        ax.tick_params(labelsize=8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Page Faults: All Queries × PCs (Boxplot)", fontsize=13, fontweight="bold", y=1.00)
plt.show()

print("✓ Faceted grid (page-faults) created")

In [ ]:
# Faceted view: Memory/Cache Events across all queries
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, len(plot_df["query_name"].unique()), hspace=0.4, wspace=0.3)

memory_events_facet = ["cache-misses", "branch-misses", "page-faults"]

for e_idx, event in enumerate(memory_events_facet):
    for q_idx, query in enumerate(queries_sorted):
        ax = fig.add_subplot(gs[e_idx, q_idx])
        
        subset = plot_df[(plot_df["event"] == event) & (plot_df["query_name"] == query)]
        
        if not subset.empty:
            sns.boxplot(
                data=subset,
                x="pc_id",
                y="value_raw",
                ax=ax,
                palette="Set2"
            )
            title = f"{query}\n{event}"
            ax.set_title(title, fontsize=9, fontweight="bold")
            ax.set_xlabel("")
            ax.set_ylabel("Value" if q_idx == 0 else "", fontsize=8)
            ax.tick_params(labelsize=7)
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_xticks([])
            ax.set_yticks([])

fig.suptitle("Memory/Cache Events: All Queries × PCs", fontsize=14, fontweight="bold", y=0.995)
plt.show()

print("✓ Faceted view (memory/cache events) created")